# Downloading and processing ERA5 for the DL downscaling task

In [ ]:
import xarray as xr

## Data source location
- `s3://nsf-ncar-era5/`

## Download invariant files

In [ ]:
!aws s3 cp --no-sign-request s3://nsf-ncar-era5/e5.oper.invariant/197901/e5.oper.invariant.128_172_lsm.ll025sc.1979010100_1979010100.nc ERA5/era5_full_lsm.nc

In [ ]:
!aws s3 cp --no-sign-request s3://nsf-ncar-era5/e5.oper.invariant/197901/e5.oper.invariant.128_129_z.ll025sc.1979010100_1979010100.nc ERA5/era5_full_z.nc

## Examine the full data

In [ ]:
# Open file safely
with xr.open_dataset('ERA5/era5_full_lsm.nc') as ds:
    lsm = ds.LSM.isel(time=0).drop_vars('time')
    lsm = lsm.load()   # ensure data is in memory

# Now the file is closed → safe to overwrite
lsm.to_netcdf('ERA5/era5_full_lsm.nc', mode='w')

# Plot
lsm.plot(x='longitude', y='latitude')

In [ ]:
import xarray as xr

z = xr.open_dataset('ERA5/era5_full_z.nc').Z
z = z.isel(time=0).drop_vars('time')

# Convert geopotential to height
g = 9.80665  # m s^-2
orog = z / g

# Clip to non-negative values
orog = orog.clip(min=0)

# Rename variable
orog = orog.rename("orog")

# Update attributes
orog.attrs["units"] = "m"
orog.attrs["long_name"] = "Orography (geopotential height)"
orog.attrs["comment"] = "Clipped to non-negative values"

# Save
orog.to_netcdf('ERA5/era5_full_orography.nc')

# Plot
orog.plot(x='longitude', y='latitude')

## Crop ERA5 to NYS boundary, same as the GFS indices

In [ ]:
orog = xr.open_dataset('ERA5/era5_full_orography.nc').orog
orog.sel(latitude=slice(48,38),longitude=slice(278,292)).plot.contourf(x='longitude', y='latitude')

- The ERA5 orography is almost identical to the GFSp25

In [ ]:
orog_nys_cropped = orog.sel(latitude=slice(48,38),longitude=slice(278,292))
orog_nys_cropped.to_netcdf('ERA5/era5_nys_cropped_orography.nc')

In [ ]:
lsm = xr.open_dataset('ERA5/era5_full_lsm.nc').LSM
lsm.sel(latitude=slice(48,38),longitude=slice(278,292)).plot.contourf(x='longitude', y='latitude')

In [ ]:
lsm_nys_cropped = lsm.sel(latitude=slice(48,38),longitude=slice(278,292))
lsm_nys_cropped.to_netcdf('ERA5/era5_nys_cropped_lsm.nc')

## Creating xesmf regridders

In [ ]:
era5_nys_cropped = xr.open_dataset("ERA5/era5_nys_cropped_orography.nc")
urma_nys = xr.open_dataset("urma_nys_orography.nc")

### Full HR grid (256 x 288)

In [ ]:
import xesmf as xe
# We don't have to provide grid dicts, rather source files having the latitude/lat and longitude/lon coordinates can work.
regridder_era5_to_urma = xe.Regridder(
    era5_nys_cropped,
    urma_nys,
    method='bilinear',
    filename='ERA5/xesmf_weights_era5_to_urma_full_HR.nc',
    reuse_weights=False
)

In [ ]:
regridded = regridder_era5_to_urma(era5_nys_cropped.orog)
print(regridded.sizes)
regridded.plot.contourf(x='longitude', y='latitude')

### Full LR grid (256/12 x 288/8)

In [ ]:
sy = 12
sx = 9
target_grid = urma_nys.isel(
    y=slice(0,None,sy),
    x=slice(0,None,sx)
).copy()   # forces contiguous array
regridder_era5_to_urma = xe.Regridder(
    era5_nys_cropped,
    target_grid,
    method='bilinear',
    filename=f'ERA5/xesmf_weights_era5_to_urma_full_LR_{sy}x{sx}.nc',
    reuse_weights=False
)

In [ ]:
regridded = regridder_era5_to_urma(era5_nys_cropped.orog)
print(regridded.sizes)
regridded.plot.contourf(x='longitude', y='latitude')